# PhysiCar ROS 2

ROS 2 interfaces and the Web API of the PhysiCar robot — same layout as the physicar-ros README.

## [1] ROS 2 Interfaces

### 1.0. Setup

One node shared by every example in this notebook. Re-running is safe — `rclpy.init()` runs only once per kernel.

In [ ]:
import json
import math
import time

import requests

import rclpy
from rclpy.qos import qos_profile_sensor_data
from rclpy.wait_for_message import wait_for_message
from IPython.display import Image, display
from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry
from sensor_msgs.msg import BatteryState, CompressedImage, Imu, LaserScan
from std_msgs.msg import Float64

rclpy.init()

node = rclpy.create_node("tutorial")

BASE_URL = "http://localhost"

### 1.1. Sensors

`wait_for_message` receives a single message without spinning — ideal for notebooks.

#### 1.1.1. Camera

`/camera/image_raw/compressed` · [`CompressedImage`](https://docs.ros2.org/latest/api/sensor_msgs/msg/CompressedImage.html)

Camera image (JPEG) — the payload is a ready-to-display JPEG, no decoding needed.

In [ ]:
ok, image = wait_for_message(CompressedImage, node, "/camera/image_raw/compressed", time_to_wait=2.0)
display(Image(data=bytes(image.data)))

#### 1.1.2. Battery

`/battery_state` · [`BatteryState`](https://docs.ros2.org/latest/api/sensor_msgs/msg/BatteryState.html)

Battery state (1 Hz). `percentage` is a 0–1 ratio.

In [ ]:
ok, battery = wait_for_message(BatteryState, node, "/battery_state", time_to_wait=3.0)
print(f"voltage: {battery.voltage:.2f} V  charge: {battery.percentage:.0%}")

#### 1.1.3. IMU

`/imu` · [`Imu`](https://docs.ros2.org/latest/api/sensor_msgs/msg/Imu.html)

IMU (50 Hz).

In [ ]:
ok, imu = wait_for_message(Imu, node, "/imu", time_to_wait=2.0)
av, la = imu.angular_velocity, imu.linear_acceleration
print(f"angular velocity    : x={av.x:.3f}  y={av.y:.3f}  z={av.z:.3f} rad/s")
print(f"linear acceleration : x={la.x:.3f}  y={la.y:.3f}  z={la.z:.3f} m/s^2")

#### 1.1.4. Odometry

`/odom` · [`Odometry`](https://docs.ros2.org/latest/api/nav_msgs/msg/Odometry.html)

Odometry.

In [ ]:
ok, odom = wait_for_message(Odometry, node, "/odom", time_to_wait=2.0)
pos, vel = odom.pose.pose.position, odom.twist.twist
print(f"position : x={pos.x:.3f}  y={pos.y:.3f}")
print(f"velocity : linear={vel.linear.x:.3f} m/s  angular={vel.angular.z:.3f} rad/s")

#### 1.1.5. LiDAR (raw)

`/scan` · [`LaserScan`](https://docs.ros2.org/latest/api/sensor_msgs/msg/LaserScan.html)

LiDAR scan (raw). LiDAR publishes with best-effort sensor QoS — the subscription must pass `qos_profile_sensor_data` to match.

In [ ]:
ok, scan = wait_for_message(LaserScan, node, "/scan",
                            qos_profile=qos_profile_sensor_data, time_to_wait=3.0)
print([round(r, 3) for r in scan.ranges])

#### 1.1.6. LiDAR (filtered)

`/scan_filtered` · [`LaserScan`](https://docs.ros2.org/latest/api/sensor_msgs/msg/LaserScan.html)

LiDAR scan (filtered).

In [ ]:
ok, scan_filtered = wait_for_message(LaserScan, node, "/scan_filtered",
                                     qos_profile=qos_profile_sensor_data, time_to_wait=3.0)
print([round(r, 3) for r in scan_filtered.ranges])

### 1.2. Control

Publishing needs a discovered subscriber — each cell waits for the driver before sending (a message published before matching is lost).

#### 1.2.1. Velocity + Steering

`/cmd_vel` · [`Twist`](https://docs.ros2.org/latest/api/geometry_msgs/msg/Twist.html)

Velocity + steering (Ackermann conversion): `linear.x` is speed (m/s), `angular.z` is turn rate (rad/s). Drive commands expire after ~1 s (`cmd_timeout`) without renewal — the speed stops but the wheels keep their angle. A zero `Twist()` is an explicit stop that also recenters the steering.

In [ ]:
cmd_vel_pub = node.create_publisher(Twist, "/cmd_vel", 10)
while cmd_vel_pub.get_subscription_count() == 0:
    time.sleep(0.1)

msg = Twist()
msg.linear.x = 0.5
msg.angular.z = 0.5
cmd_vel_pub.publish(msg)

#### 1.2.2. Speed

`/speed` · [`Float64`](https://docs.ros2.org/latest/api/std_msgs/msg/Float64.html)

Speed (m/s) — commands expire after the driver's `cmd_timeout` (default 1 s, `0` disables) without renewal; publish periodically for sustained driving.

In [ ]:
speed_pub = node.create_publisher(Float64, "/speed", 10)
while speed_pub.get_subscription_count() == 0:
    time.sleep(0.1)

speed_pub.publish(Float64(data=0.5))

#### 1.2.3. Steering

`/steering` · [`Float64`](https://docs.ros2.org/latest/api/std_msgs/msg/Float64.html)

Steering angle (rad) — + = left, max ±20° (±0.35 rad).

In [ ]:
steering_pub = node.create_publisher(Float64, "/steering", 10)
while steering_pub.get_subscription_count() == 0:
    time.sleep(0.1)

steering_pub.publish(Float64(data=math.radians(10)))

#### 1.2.4. Camera Pan

`/camera/pan` · [`Float64`](https://docs.ros2.org/latest/api/std_msgs/msg/Float64.html)

Camera pan (rad) — + = left, range ±30° (±0.52 rad).

In [ ]:
pan_pub = node.create_publisher(Float64, "/camera/pan", 10)
while pan_pub.get_subscription_count() == 0:
    time.sleep(0.1)

pan_pub.publish(Float64(data=math.radians(15)))

#### 1.2.5. Camera Tilt

`/camera/tilt` · [`Float64`](https://docs.ros2.org/latest/api/std_msgs/msg/Float64.html)

Camera tilt (rad) — + = up, range ±30° (±0.52 rad).

In [ ]:
tilt_pub = node.create_publisher(Float64, "/camera/tilt", 10)
while tilt_pub.get_subscription_count() == 0:
    time.sleep(0.1)

tilt_pub.publish(Float64(data=math.radians(15)))

## [2] Web API

The same interfaces over HTTP — interactive docs at `/docs` (OpenAPI).

### 2.1. Sensor Queries

Query endpoints support real-time streaming via `?stream=true` (camera uses MJPEG, others use SSE).

#### 2.1.1. States

`GET /states` — full state snapshot, select with `?include=odom,battery,imu`.

In [ ]:
requests.get(f"{BASE_URL}/states", params={"include": "odom,battery"}).json()

#### 2.1.2. Speed

`GET /speed` — speed (m/s).

In [ ]:
requests.get(f"{BASE_URL}/speed").json()

#### 2.1.3. Steering

`GET /steering` — steering angle (rad).

In [ ]:
requests.get(f"{BASE_URL}/steering").json()

#### 2.1.4. Odometry

`GET /odom`

In [ ]:
requests.get(f"{BASE_URL}/odom").json()

#### 2.1.5. Battery

`GET /battery`

In [ ]:
requests.get(f"{BASE_URL}/battery").json()

#### 2.1.6. IMU

`GET /imu`

In [ ]:
requests.get(f"{BASE_URL}/imu").json()

#### 2.1.7. LiDAR

`GET /lidar` — scan with `ranges`, `range_min`/`range_max`, `count`.

In [ ]:
lidar = requests.get(f"{BASE_URL}/lidar").json()
print(lidar["count"], "points")
print(lidar["ranges"])

#### 2.1.8. Camera

`GET /camera` — camera image (JPEG), resize with `?width`/`?height`.

In [ ]:
jpg = requests.get(f"{BASE_URL}/camera", params={"width": 480}).content
display(Image(data=jpg))

#### 2.1.9. Camera Pan

`GET /camera/pan` — pan angle (rad).

In [ ]:
requests.get(f"{BASE_URL}/camera/pan").json()

#### 2.1.10. Camera Tilt

`GET /camera/tilt` — tilt angle (rad).

In [ ]:
requests.get(f"{BASE_URL}/camera/tilt").json()

### 2.2. Control

POST the same `{"value": ...}` body everywhere. Speed follows the same `cmd_timeout` contract as the `/speed` topic.

#### 2.2.1. Speed

`POST /speed` — `{"value": m/s, "duration": seconds?}`. Without `duration` it expires after `cmd_timeout` (~1 s) unless renewed. With `duration` the server keeps the command alive, publishes 0 at the end, and the response returns after the drive finishes.

In [ ]:
requests.post(f"{BASE_URL}/speed", json={"value": 0.5, "duration": 2.0}, timeout=10).json()

#### 2.2.2. Steering

`POST /steering` — steering angle (rad), persists until changed.

In [ ]:
requests.post(f"{BASE_URL}/steering", json={"value": math.radians(10)}).json()

#### 2.2.3. Camera Pan

`POST /camera/pan`

In [ ]:
requests.post(f"{BASE_URL}/camera/pan", json={"value": math.radians(15)}).json()

#### 2.2.4. Camera Tilt

`POST /camera/tilt`

In [ ]:
requests.post(f"{BASE_URL}/camera/tilt", json={"value": math.radians(15)}).json()

#### 2.2.5. Streaming write (WebSocket)

`WS /speed/stream`, `/steering/stream` — each frame is the same `{"value": x}` as the POST. Dead-man switch: on disconnect the value is zeroed, so a dead client can never leave the robot driving.

In [ ]:
from websockets.sync.client import connect

with connect("ws://localhost/speed/stream") as ws:
    for _ in range(20):  # ~2 s of driving at 10 Hz
        ws.send(json.dumps({"value": 0.5}))
        time.sleep(0.1)
# socket closed -> the dead-man switch zeroes the speed

### 2.3. Audio

Command-based playback on the robot speaker (played in the browser viewer in SIM).

#### 2.3.1. Play

`POST /audio/play` — one of `url` / `path` / `data` (base64). Options: `volume` (0–1), `loop`, `replace`.

In [ ]:
import struct
import wave

path = "/tmp/beep.wav"
with wave.open(path, "w") as f:
    f.setnchannels(1)
    f.setsampwidth(2)
    f.setframerate(24000)
    f.writeframes(b"".join(struct.pack("<h", int(8000 * math.sin(2 * math.pi * 440 * t / 24000)))
                           for t in range(24000)))  # 1 s, 440 Hz

requests.post(f"{BASE_URL}/audio/play", json={"path": path, "volume": 0.8}).json()

#### 2.3.2. Now playing

`GET /audio`

In [ ]:
requests.get(f"{BASE_URL}/audio").json()

#### 2.3.3. Stop

`POST /audio/stop` — by `id`, or everything with `{"all": true}`.

In [ ]:
requests.post(f"{BASE_URL}/audio/stop", json={"all": True}).json()